# 23 homa budget

Question: do the headers, the memory, the latency and the goodput of the crate `homa` stay inside
the budget of the Homa design note, second version, section 8, and what does Homa cost against
AoFrame alone.

The simulated profiles run through `aosabi-devtool homa-sim` over the loopback and the baseline of
AoFrame alone of `homa::sim`, the sizes of the types come from its operation `sizes`, and the device
image is measured through `tests/fixtures/rs/homa-probe`, a static library for
`riscv32imc-unknown-none-elf`, the RV32IMC core of the ESP32-C3, which instantiates the endpoint of
a card with the configuration of the design note, second version, section 8.3.

The notebook runs one half per entry of the matrix, which `half` names. The simulation half with the
probe cell is stage H0's. The client half is stage H2's, the cells of the page's client role in
`tests/fixtures/rs/wasm-client-probe`, a module for `wasm32-unknown-unknown`, and of the bytes an
answer in progress holds through `reply_reserve` against `reply` with a buffer. The half
`raw-server` is the raw server cell of stage H4.

A row that crosses its bound fails the notebook, since a crossed bound fails the change that crosses
it and the owner decides whether the bound moves. The owner answered both questions, so no row of
the simulation half prints OPEN.

> The row prints OPEN beside its measure and the gzip size, the verdict does not fail on it, every
> other row keeps its assert, and the row returns to an assert with the bound the owner names. The
> lead put three options to the owner: A (the lead's recommendation) judge the browser client on its
> compressed size at most 32 KiB, B raise the raw bound for the browser to 64 KiB, C keep 32 KiB raw
> and shrink the client role in a later batch.

So the row of the wasm client role prints OPEN with its bytes and its gzip size, the entry
`23_homa_budget-client` does not fail on it, and every other row of the client cells keeps its
assert.

The exit number of the LoRa probe cell, stage H6: at most 32 KiB, 32,768 bytes, for `homa` with
`pacer-budget` and AoFrame, at most 18 KiB for LoRa, 4 KiB for the driver of the SX1262, 2 KiB for
the frame, the ledger and the time on air and 12 KiB for `chacha20poly1305`, and `graphtp` and
`graphtp-homa` printed beside. The plan's section The budget gives the row of `homa` 14.5 KiB, the
13.5 KiB of the ESP32-C3 and 1 KiB of `pacer-budget`, which the design note's section 8.2 adds, and
the row prints that bound beside its measure.

> The row holds at 27,278 bytes, 5,490 bytes under the bound.

So the row of `homa` asserts against 32 KiB, and a row that crosses its bound fails the notebook.
The owner may name another bound for the row in place of the lead's.

The exit number of the raw server cell, stage H4: at most 40 KiB, 40,960 bytes, for `homa` with the
IP driver and at most 8 KiB, 8,192 bytes, for the Network driver `aosabi-carrier-network`, each the
code and read only bytes of the symbols of the crate in the release build, at most 202 bytes a peer
and 216 bytes an RPC without its pages, the state of 4,096 peers and 16,384 RPCs and the page pool
printed, the bytes of `socket2` printed as the Network carrier's own, and the size of the helper
`homa-raw` with the state of its client role printed.

> The owner may still choose B, C or D.

Judged stages: H0, H2, H4 and H6 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

half = "simulation"

# The seed of the drawn arrival times.
seed = 20260927

# The draws of the wait of a small response behind 16 KiB, on each profile
# and under each scheduling.
draws = 40

# The RPCs of the 1 percent loss of the Serial Line profile.
loss_rpcs = 500

# The bounds of the exit number. It holds the server image and the image with
# the roles client and server.
probe_budget = 32_768
peer_budget = 96
slot_budget = 72
srpt_wait_budget_ms = 60
stream_order_floor_ms = 700
transfer_slack = 0.10
speedup_floor = 10

wasm_budget = 32_768
wasm_question = 17
reserve_answer = 8192

encoder_budgets = "esp32c3-devkitm1:22528,player:53248,server:53248"

# The bounds of the LoRa probe cell of the HT-CT62.
lora_homa_budget = 32_768
lora_plan_bound = 14_848
lora_budget = 18_432
sx1262_budget = 4_096
lora_frame_budget = 2_048
aead_budget = 12_288
lora_question = 18

# The bounds of the raw server cell of stage H4, a server build that names
# GraphTP(Homa) on Network over raw IP 146: at most 40 KiB for homa with the
# IP driver and at most 8 KiB for the Network driver, read from the sums of
# the symbols of the release build, and at most 202 bytes a peer and 216
# bytes an RPC without its pages, the plan's section The budget, the row of
# the raw path. The bytes of socket2, the size of the helper homa-raw and the
# state of its client role are printed with no bound.
homa_ip_budget = 40_960
network_driver_budget = 8_192
network_peer_budget = 202
network_rpc_budget = 216

# A question named here prints its row OPEN beside its measure, and the
# verdict does not fail on it. With a question None its row asserts again, and
# an answer that moves a bound names the bound above.
driver_question = None
rpc_question = None

# The per command timeout, and the one of a cold build of the probe.
timeout_s = 600
build_timeout_s = 900

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain, the installed targets, which hold `riscv32imc-unknown-none-elf` for
the probe, the tools the probe cell reads the static library with, `llvm-size` and `llvm-nm` of the
development machine, and the gates of the plan's section The commands that the notebook rests on:
the build of the devtool, the lints of the workspace, the format of the crates `homa` and
`aosabi-devtool`, and the build of `homa` for the device with the features of the ESP32-C3 card,
every cargo call with `--locked`.

In [ ]:
import json
import os
import random
import re
import shutil
import statistics
import sys
import tomllib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(aosabi_root)
OUT = P.out / "23_homa_budget"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTFMT = CARGO_BIN / "rustfmt"
LLVM_SIZE = shutil.which("llvm-size")
LLVM_NM = shutil.which("llvm-nm")

RUSTUP = CARGO_BIN / "rustup"
PROBE_TARGET = "riscv32imc-unknown-none-elf"
DEVICE_FEATURES = "server,wire-compact,stream-delimited,stream-prefixed"
WASM_FEATURES = "client,alloc,wire-compact,stream-delimited,stream-prefixed"

# The gates of the plan's section The commands that this notebook rests on,
# each cargo call with --locked: the build of the seam it drives, the lints of
# the workspace, the build of homa for the device, and the format of the crate
# homa under its own edition 2021 and of the devtool, which rustfmt checks
# because cargo fmt takes no --locked.
GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("cargo build --locked -p homa --release, the device",
     [CARGO, "build", "--locked", "-p", "homa", "--release", "--target", PROBE_TARGET,
      "--no-default-features", "--features", DEVICE_FEATURES]),
    ("cargo build --locked -p homa --release, the page",
     [CARGO, "build", "--locked", "-p", "homa", "--release", "--target", "wasm32-unknown-unknown",
      "--no-default-features", "--features", WASM_FEATURES]),
    ("rustfmt --check --edition 2021 crates/homa/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2021", P.root / "crates" / "homa" / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
]
TARGETS = run([RUSTUP, "target", "list", "--installed"], cwd=P.root, timeout_s=60).out.split()
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=build_timeout_s, check=False)
    print("gate %-66s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "homa-sim", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("cargo          :", reported_cargo)
print("targets        :", ", ".join(TARGETS))
print("seam           :", json.dumps(smoke))
print("llvm-size      :", " ".join(run([LLVM_SIZE, "--version"], timeout_s=60).out.split()[-3:]) if LLVM_SIZE else "MISSING")
print("llvm-nm        :", "found" if LLVM_NM else "MISSING")
assert not GATE_FAILURES, GATE_FAILURES
assert half in ("simulation", "client", "graphtp", "lora", "raw-server"), half
assert PROBE_TARGET in TARGETS, "the probe is built for %s" % PROBE_TARGET
assert "wasm32-unknown-unknown" in TARGETS, "the wasm client probe is built for wasm32-unknown-unknown"
assert rust_version in reported_cargo, reported_cargo
assert smoke == {"subcommand": "homa-sim", "implemented": True, "lands_at": "H0"}, smoke
assert LLVM_SIZE and LLVM_NM, "the probe cell reads the static library with llvm-size and llvm-nm"

PANICS = []
ROWS = []
INFO = []


def row(name, value, bound, ok, mark="MEASURED", open_question=None, beside=None):
    """Record one row of the exit number.

    OPEN is a visible state and no pass, as a cell that prints not run is.
    `beside` is printed under the row here and in the verdict.
    """
    if open_question is not None:
        state = "OPEN (question %d)" % open_question
    else:
        state = "ok" if ok else "CROSSED"
    ROWS.append((name, value, bound, state, mark, beside))
    print("%-58s %-18s %-18s %s" % (name[:58], value, bound, state))
    if beside:
        print("  beside: %s" % beside)


def info(name, value, note, mark="MEASURED"):
    """Record a measure printed as information, with no bound and no assert."""
    INFO.append((name, value, note, mark))
    print("%-58s %-18s %s, %s" % (name[:58], value, note, mark))


def sim(requests, name):
    """Send requests to `homa-sim` in one process and return its answers."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run(
        [DEVTOOL, "homa-sim", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False
    )
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    bad = [a for a in answers if "fatal" in a or "config_error" in a]
    assert not bad, "the seam refused requests of %s: %s" % (name, bad[:2])
    return answers


def packet(requests, name):
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run(
        [DEVTOOL, "homa-packet", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False
    )
    return [json.loads(line) for line in result.out.splitlines()]

## The headers per carrier

The table of the design note, first version, section 6.1 gives each carrier its `max_packet`, a
header of 6 bytes for DATA that carries a whole message and 14 bytes for a part, and the overhead
of a part packet, which counts the 7 bytes the table gives AoFrame on the two delimited links. The
cell builds each profile, encodes a whole message of one byte and a part that fills the packet
with the crate's encoder, and prints the overhead by the table's own rule beside the bytes a
delimited frame of that packet takes, which COBS lengthens by one more byte at 256 and 512 bytes.

In [ ]:
if half == "simulation":
    TABLE_61 = [
        ("Unix Socket", {"name": "unix_socket", "max_message": 65536}, 16384, 0, "0.09"),
        ("USB bulk, high speed", {"name": "usb_bulk", "high_speed": True}, 16384, 0, "0.09"),
        ("USB serial", {"name": "usb_serial"}, 512, 7, "4.0"),
        ("Bluetooth GATT, ATT_MTU 185", {"name": "bluetooth_gatt", "att_mtu": 185, "interval_ns": 30_000_000}, 182, 0, "7.7"),
        ("Serial Line", {"name": "serial_line", "baud": 115200}, 256, 7, "8.0"),
    ]
    profiles = sim([{"id": i, "op": "profile", "profile": spec} for i, (_, spec, _, _, _) in enumerate(TABLE_61)], "profiles")
    header_rows_ok = 0
    print("%-28s %-10s %-6s %-6s %-10s %-10s %s" % ("carrier", "max_packet", "whole", "part", "overhead", "table", "delimited frame of the packet"))
    for (carrier, spec, table_packet, frame_bytes, table_overhead), answer in zip(TABLE_61, profiles):
        max_packet = answer["profile"]["max_packet"]
        whole = {"type": "DATA", "id": 2, "msg_length": 1, "offset": 0, "retransmit": False, "ack": None,
                 "cutoff_version": 0, "incast": False, "payload": "00"}
        body = max_packet - 14
        part = dict(whole, msg_length=body * 2, payload="ab" * body)
        encoded = packet([{"id": 0, "op": "encode", "form": "compact", "packet": whole},
                          {"id": 1, "op": "encode", "form": "compact", "packet": part}], "headers-%d" % max_packet)
        whole_header = len(bytes.fromhex(encoded[0]["hex"])) - 1
        part_len = len(bytes.fromhex(encoded[1]["hex"]))
        part_header = part_len - body
        overhead = 100.0 * (part_header + frame_bytes) / (max_packet + frame_bytes)
        printed = ("%.2f" if overhead < 1 else "%.1f") % overhead
        frame = run([DEVTOOL, "homa-frame", "--requests", "-"], cwd=P.root, timeout_s=60, check=False,
                    stdin=(json.dumps({"id": 0, "op": "encode", "mode": "delimited", "hex": encoded[1]["hex"]}) + "\n").encode())
        frame_len = len(bytes.fromhex(json.loads(frame.out)["hex"]))
        ok = max_packet == table_packet and whole_header == 6 and part_header == 14 and printed == table_overhead and part_len == max_packet
        header_rows_ok += ok
        print("%-28s %-10d %-6d %-6d %-10s %-10s %d bytes, %d over the packet%s" % (
            carrier, max_packet, whole_header, part_header, printed, table_overhead, frame_len,
            frame_len - part_len, "" if ok else "  DIFFERS"))
    print()
    row("headers equal the table of design-v1 section 6.1", "%d of %d" % (header_rows_ok, len(TABLE_61)), "5 of 5", header_rows_ok == len(TABLE_61))

## The memory of a peer, a slot and a card

The design note, second version, section 8.3 budgets a peer at 96 bytes and a slot at 72 bytes
plus its pages, in place of the 80 and 64 of the first version, section 6.3. The devtool prints
the sizes of the types on this host, which match the RV32IMC core, since no type of the engine
holds a pointer and both align a `u64` to 8 bytes. The peer the budget counts is the handle of a
point to point link of the card, which the probe names by the index of the link, a `u8`, and a
Unix Socket, USB and a serial line by `()`, and the simulators by `Side`. The handle of a central
of the GATT data path is a `u32` in the crate today, whose peer is printed beside, and the carrier
of stage H3 chooses the handle its driver keeps. The storage of the card of section 8.3, four
links, eight server slots and a pool of 40 pages of 512 bytes, one request page and four answer
pages a slot, is printed with the buffers of AoFrame on its links.

In [ ]:
if half == "simulation":
    DEVICE = {"peers": 4, "client_slots": 0, "server_slots": 8, "page_len": 512, "pages": 40, "max_gaps": 2}
    sizes = sim([{"id": 0, "op": "sizes", "config": DEVICE, "max_packets": [256, 512, 488, 182, 16384]}], "sizes")[0]
    print("slot:", sizes["slot"], "bytes")
    print("peer:", sizes["peer"])
    print("bucket:", sizes["bucket"], " heap item:", sizes["item"], " endpoint beside its storage:", sizes["endpoint_u8"])
    print("device storage parts:", sizes["storage"])
    print("device storage bytes:", sizes["storage_bytes"])
    buffers = {b["max_packet"]: b for b in sizes["buffers"]}
    aoframe_device = (
        buffers[256]["delimited"] + buffers[256]["frame_delimited"]
        + buffers[512]["delimited"] + buffers[512]["frame_delimited"]
        + buffers[488]["prefixed"] + buffers[488]["frame_prefixed"]
    )
    print("AoFrame of the card, a decoder and an encoder on Serial Line, USB serial and the channel:", aoframe_device, "bytes")
    largest_peer = max(sizes["peer"][k] for k in ("unit", "u8", "u16", "side"))
    row("a peer of a link of the card, bytes", largest_peer, "at most %d" % peer_budget, largest_peer <= peer_budget)
    print("a peer whose handle is a u32, the central of the GATT data path today:", sizes["peer"]["u32"], "bytes, the choice of stage H3")
    row("a slot without its pages, bytes", sizes["slot"], "at most %d" % slot_budget, sizes["slot"] <= slot_budget)
    print("a peer of Network, an address and a port:", sizes["peer"]["socket_addr"], "bytes, the budget of stage H4")

## A slot of more than 16 pages

One message spans at most 16 pages of the pool, so a pool that gives a slot more than 16 pages means
pages too short for the messages it is sized for, and `Endpoint::new` refuses it with
`ConfigError::TooSmall`.

In [ ]:
if half == "simulation":
    built = sim([
        {"id": 0, "op": "build", "profile": {"name": "serial_line"},
         "config": {"peers": 1, "client_slots": 1, "server_slots": 1, "page_len": 64, "pages": 33}},
        {"id": 1, "op": "build", "profile": {"name": "serial_line"},
         "config": {"peers": 1, "client_slots": 1, "server_slots": 1, "page_len": 64, "pages": 32}},
    ], "too-small")
    print("33 pages for 2 slots:", built[0])
    print("32 pages for 2 slots:", built[1])
    row("a slot of more than 16 pages refused", built[0].get("error"), "TooSmall",
        built[0].get("error") == "TooSmall" and built[1].get("ok") is True)

## The wait of a small response behind 16 KiB

The design note, first version, section 6.4 puts a small request on the link while the server sends
a response of 16 KiB, at a moment spread uniformly over that response, and asks how long the small
response waits, from the server holding the small request to the client holding its answer. Under
SRPT it waits for the connection event that runs and the one that carries the two packets queued
below the engine, 45 ms on average on GATT, and in first in first out order it waits for the rest of
the large response, 0.735 s on average. A calibration run of each profile gives the time the large
response takes and the time the small request takes to reach the server, and each draw sends the
small request at a seeded time inside that span, moving the clock there. The same draws run with the
FIFO share of 1, where every choice is first in first out, and over the baseline of AoFrame alone.
On the two delimited profiles, USB serial and the Serial Line, the link times every packet of Homa
by the bytes of its delimited AoFrame, as the byte stream carries it and as the baseline times its
own frames.

In the byte stream order the design note compares SRPT with, the small response waits for the rest
of the large one, so the same draws run the FIFO share of 1 on a sender whose large message never
waits for a grant, the whole message unscheduled and a window of the whole message, with SRPT on the
same link beside it. The FIFO share of 1 on the profile as it stands lets the small response past
the large one whenever the large one has sent every byte it was granted and waits for the next
GRANT, so there the small response waits for a window at most and not for the rest of the large
response. That wait is printed as information with no bound, and so are the FIFO share of 1 and the
byte stream order on the channel profile, where the whole large response takes less than 0.7 s. SRPT
on the channel keeps its bound of 60 ms.

The 16 KiB transfer runs from the server holding the large request to the client holding the
whole response, and its packet time is the time the link model takes for its packets: whole
connection events of two packets on GATT and the channel, and the bytes of every packet at the
line rate on the Serial Line, each packet in the delimited frame the crate writes for a packet
with no zero byte, which is the longest COBS makes it and what the design note, first version,
section 6.4 counts.

In [ ]:
if half == "simulation":
    PROFILES = {
        "Unix Socket": ({"name": "unix_socket", "max_message": 65536},
                        {"bytes_per_s": 0, "packets_per_event": 2, "event_ns": 0, "latency_ns": 10_000}, "prefixed"),
        "USB bulk": ({"name": "usb_bulk", "high_speed": True},
                     {"bytes_per_s": 40_000_000, "packets_per_event": 2, "event_ns": 0, "latency_ns": 125_000}, "prefixed"),
        "USB serial": ({"name": "usb_serial"},
                       {"bytes_per_s": 1_000_000, "packets_per_event": 2, "event_ns": 0, "latency_ns": 1_000_000}, "delimited"),
        "Bluetooth GATT": ({"name": "bluetooth_gatt", "att_mtu": 185, "interval_ns": 30_000_000},
                           {"bytes_per_s": 12_133, "packets_per_event": 2, "event_ns": 30_000_000, "latency_ns": 1_000_000}, "prefixed"),
        "Bluetooth channel": ({"name": "bluetooth_channel", "peer_mps": 247, "interval_ns": 30_000_000},
                              {"bytes_per_s": 36_000, "packets_per_event": 2, "event_ns": 30_000_000, "latency_ns": 1_000_000}, "prefixed"),
        "Serial Line": ({"name": "serial_line", "baud": 115200},
                        {"bytes_per_s": 11_520, "packets_per_event": 1, "event_ns": 0, "latency_ns": 100_000}, "delimited"),
    }
    CONFIG = {"client_slots": 8, "server_slots": 8, "page_len": 4096, "pages": 256, "max_gaps": 2}
    START = 200_000_000
    BIG = {"len": 3, "response_len": 16384}
    PAUSE_REQUEST = len(b'{"query":"mutation{pause}"}')
    PAUSE_ANSWER = len(b'{"data":{"pause":true}}')


    def run_body(ident, name, rpcs, link="loopback", overrides=None, faults=None):
        spec, model, framing = PROFILES[name]
        body = {"id": ident, "op": "run", "link": link, "profile": spec, "model": model, "config": CONFIG,
                "window": 4, "seed": seed + ident, "rpcs": rpcs, "baseline_mode": framing,
                "frame": framing if framing == "delimited" else "none"}
        if overrides:
            body["overrides"] = overrides
        if faults:
            body["faults"] = faults
        return body


    calibration = sim(
        [run_body(i, name, [dict(BIG, at_ns=START)]) for i, name in enumerate(PROFILES)]
        + [run_body(100 + i, name, [{"len": PAUSE_REQUEST, "response_len": PAUSE_ANSWER, "at_ns": START}]) for i, name in enumerate(PROFILES)],
        "calibration",
    )
    SPAN = {}
    for i, name in enumerate(PROFILES):
        big = calibration[i]["rpcs"][0]
        small = calibration[len(PROFILES) + i]["rpcs"][0]
        SPAN[name] = {
            "big_start": big["request_ns"], "big_end": big["response_ns"],
            "uplink": small["request_ns"] - small["issued_ns"],
            "link_bytes_big": calibration[i]["links"]["B"],
            "link_bytes_pause_up": calibration[len(PROFILES) + i]["links"]["A"],
            "max_packet": calibration[i]["profile"]["max_packet"],
        }
    g = random.Random(seed)
    plans = {name: [g.uniform(0, 1) for _ in range(draws)] for name in PROFILES}


    def draw_rpcs(name, where):
        span = SPAN[name]
        at = int(span["big_start"] + where * (span["big_end"] - span["big_start"]) - span["uplink"])
        return [dict(BIG, at_ns=START), {"len": PAUSE_REQUEST, "response_len": PAUSE_ANSWER, "at_ns": max(at, START)}]


    bodies = []
    index = []
    OPEN = {"unsched_bytes": 65536, "window": 65536}
    MODES = {
        "srpt": None,
        "fifo": {"fifo_every": 1},
        "srpt open": dict(OPEN),
        "fifo open": dict(OPEN, fifo_every=1),
        "baseline": None,
    }
    for name in PROFILES:
        for mode, overrides in MODES.items():
            for where in plans[name]:
                link = "baseline" if mode == "baseline" else "loopback"
                bodies.append(run_body(1000 + len(bodies), name, draw_rpcs(name, where), link=link, overrides=overrides))
                index.append((name, mode))
    answers = sim(bodies, "waits")
    WAITS = {}
    TRANSFER = {}
    for (name, mode), answer in zip(index, answers):
        big, small = answer["rpcs"]
        assert big["outcome"] == "ok" and small["outcome"] == "ok", (name, mode, big["outcome"], small["outcome"])
        WAITS.setdefault((name, mode), []).append((small["response_ns"] - small["request_ns"]) / 1e6)
        TRANSFER.setdefault((name, mode), []).append((big["response_ns"] - big["request_ns"]) / 1e6)


    def packet_lengths(name):
        """The lengths of the packets of a response of 16 KiB, parts with a
        header of 14 bytes each."""
        body = SPAN[name]["max_packet"] - 14
        count = -(-16384 // body)
        return [14 + min(body, 16384 - i * body) for i in range(count)]


    # The delimited frame of each packet, as the crate writes it for a packet with
    # no zero byte, read from homa-frame.
    FRAMED = {}
    for name, (spec, model, framing) in PROFILES.items():
        if framing != "delimited":
            continue
        lengths = packet_lengths(name)
        request = "".join(json.dumps({"id": i, "op": "encode", "mode": "delimited", "hex": "ab" * n}) + "\n"
                          for i, n in enumerate(lengths))
        framed = run([DEVTOOL, "homa-frame", "--requests", "-"], cwd=P.root, timeout_s=60, stdin=request.encode())
        FRAMED[name] = [len(bytes.fromhex(json.loads(line)["hex"])) for line in framed.out.splitlines()]
        assert len(FRAMED[name]) == len(lengths), name


    def packet_time_ms(name):
        spec, model, framing = PROFILES[name]
        lengths = packet_lengths(name)
        if model["event_ns"]:
            return -(-len(lengths) // model["packets_per_event"]) * model["event_ns"] / 1e6
        if not model["bytes_per_s"]:
            return 0.0
        if framing == "delimited":
            lengths = FRAMED[name]
        return sum(lengths) / model["bytes_per_s"] * 1e3


    print("mean waits of the small response behind 16 KiB, ms, and the 16 KiB transfer")
    print("%-18s %-8s %-8s %-8s %-10s %-10s %-10s %-10s %s" % (
        "profile", "SRPT", "worst", "FIFO 1", "SRPT open", "FIFO open", "AoFrame", "16 KiB", "packet time"))
    for name in PROFILES:
        srpt_waits = WAITS[(name, "srpt")]
        print("%-18s %-8.1f %-8.1f %-8.1f %-10.1f %-10.1f %-10.1f %-10.1f %.1f" % (
            name, statistics.mean(srpt_waits), max(srpt_waits), statistics.mean(WAITS[(name, "fifo")]),
            statistics.mean(WAITS[(name, "srpt open")]), statistics.mean(WAITS[(name, "fifo open")]),
            statistics.mean(WAITS[(name, "baseline")]), statistics.mean(TRANSFER[(name, "srpt")]),
            packet_time_ms(name)))
    print()
    gatt_srpt = statistics.mean(WAITS[("Bluetooth GATT", "srpt")])
    gatt_stream = statistics.mean(WAITS[("Bluetooth GATT", "fifo open")])
    channel_srpt = statistics.mean(WAITS[("Bluetooth channel", "srpt")])
    OWNER_Q13 = "the owner, 2026-09-27, on owner question 13: \"Q13 just use Homa normal A\""
    row("SRPT mean wait on the GATT profile, ms", "%.1f" % gatt_srpt, "at most %d" % srpt_wait_budget_ms, gatt_srpt <= srpt_wait_budget_ms)
    row("byte stream order, mean wait on the GATT profile, ms", "%.1f" % gatt_stream, "at least %d" % stream_order_floor_ms,
        gatt_stream >= stream_order_floor_ms,
        beside="SRPT %.1f ms against it on the same profile, %s" % (gatt_srpt, OWNER_Q13))
    row("SRPT mean wait on the channel profile, ms", "%.1f" % channel_srpt, "at most %d" % srpt_wait_budget_ms, channel_srpt <= srpt_wait_budget_ms)
    info("FIFO share of 1, mean wait on the GATT profile, ms", "%.1f" % statistics.mean(WAITS[("Bluetooth GATT", "fifo")]),
         "no bound, the profile as it stands, a window at most")
    info("FIFO share of 1, mean wait on the channel profile, ms", "%.1f" % statistics.mean(WAITS[("Bluetooth channel", "fifo")]),
         "no bound, the rest of the large response")
    info("byte stream order, mean wait on the channel profile, ms", "%.1f" % statistics.mean(WAITS[("Bluetooth channel", "fifo open")]),
         "no bound, the whole 16 KiB response takes %.1f ms there" % statistics.mean(TRANSFER[("Bluetooth channel", "srpt")]))
    info("SRPT with no grant gap, mean wait on the GATT profile, ms", "%.1f" % statistics.mean(WAITS[("Bluetooth GATT", "srpt open")]),
         "no bound")
    for name in ("Bluetooth GATT", "Bluetooth channel", "Serial Line"):
        transfer = statistics.mean(TRANSFER[(name, "srpt")])
        bound = packet_time_ms(name) * (1 + transfer_slack)
        row("16 KiB transfer on %s, ms" % name, "%.1f" % transfer, "at most %.1f" % bound, transfer <= bound)

## GraphTP(Homa) against AoFrame alone

ADR-0023 asks the plan that owns the carriers to measure the two on each carrier in bytes on the
link, latency, throughput, bytes added to each target and resident memory, and the table of the
design note, first version, section 6.6 names what each row holds. At stage H0 no GraphTP travels,
so the Homa column carries the bytes a GraphTP message would take as Homa RPCs of the same length,
and the AoFrame column the same bytes as whole messages in AoFrame alone over the baseline, first in
first out with no Homa header and no retransmission. The bytes of `graphtp` and `graphtp-homa` and
the memory their listener takes per card are the half `graphtp` of this notebook, an entry of the
matrix of its own.

The bytes on the link are those of a pause request of 27 bytes up and of the response of 16 KiB
down: the DATA packets of Homa, header and payload, against AoFrame's frame of the whole message,
and every byte each end put on the link, the LIMITS, the GRANTs and the acknowledgements of Homa
and the frames of the delimited links included, is printed below the table. The latency is the
mean wait of the section above, where the link times every packet of Homa on the delimited
profiles by its frame, as the baseline times its frames. The throughput is the goodput of the 16 KiB response. The bytes added to a target
are the probe's, below, and the resident memory is the storage of the card of section 8.3 against
one frame buffer each way on each link for AoFrame alone.

In [ ]:
if half == "simulation":
    TABLE_66 = {}
    for name in PROFILES:
        span = SPAN[name]
        baseline_calibration = sim([
            run_body(0, name, [dict(BIG, at_ns=START)], link="baseline"),
            run_body(1, name, [{"len": PAUSE_REQUEST, "response_len": PAUSE_ANSWER, "at_ns": START}], link="baseline"),
        ], "baseline-" + name.replace(" ", "-"))
        base_big = baseline_calibration[0]["rpcs"][0]
        homa_transfer = statistics.mean(TRANSFER[(name, "srpt")])
        base_transfer = (base_big["response_ns"] - base_big["request_ns"]) / 1e6
        TABLE_66[name] = {
            "bytes up, a pause": (span["link_bytes_pause_up"]["data_header_bytes"] + PAUSE_REQUEST,
                                  baseline_calibration[1]["links"]["A"]["frame_bytes"]),
            "bytes down, 16 KiB": (span["link_bytes_big"]["data_header_bytes"] + 16384,
                                   baseline_calibration[0]["links"]["B"]["frame_bytes"]),
            "wait behind 16 KiB, ms": (statistics.mean(WAITS[(name, "srpt")]), statistics.mean(WAITS[(name, "baseline")])),
            "goodput, bytes a second": (16384 / (homa_transfer / 1e3) if homa_transfer else float("inf"),
                                        16384 / (base_transfer / 1e3) if base_transfer else float("inf")),
        }

### The loss of the Serial Line profile

The Serial Line profile runs `loss_rpcs` RPCs under 1 percent loss, for Homa of every packet over
the lossy link and for AoFrame alone of every message over the baseline, which loses a request or
an answer with every damaged frame and never sends it again.

In [ ]:
if half == "simulation":
    g_loss = random.Random(seed ^ 0x23)
    loss_specs = [{"len": g_loss.randint(20, 300), "response_len": g_loss.randint(20, 600)} for _ in range(loss_rpcs)]
    homa_loss, base_loss = sim([
        dict(run_body(0, "Serial Line", loss_specs, link="lossy", faults={"drop_per_million": 10_000}), max_time_ns=3_600_000_000_000),
        run_body(1, "Serial Line", loss_specs, link="baseline", faults={"drop_per_million": 10_000}),
    ], "loss")
    homa_lost = sum(1 for got in homa_loss["rpcs"] if got["outcome"] != "ok")
    base_lost = sum(1 for got in base_loss["rpcs"] if got["outcome"] != "ok")
    print("Homa under 1 percent loss: %d of %d RPCs lost, %d RESENDs" % (
        homa_lost, loss_rpcs, homa_loss["stats"]["A"]["resends"] + homa_loss["stats"]["B"]["resends"]))
    print("AoFrame alone under 1 percent loss: %d of %d RPCs lost" % (base_lost, loss_rpcs))

## The probe for the device image

The probe is a static library for `riscv32imc-unknown-none-elf` with `homa` in the features of the
ESP32-C3 card, `server`, `wire-compact`, `stream-delimited` and `stream-prefixed`, whose functions
instantiate the endpoint of the card and the codec of AoFrame, kept in the object through the link
time optimization of its release profile by a static marked `#[used]`. The cell builds it three
times, whole, with the feature `frame-only`, which keeps AoFrame alone, and with the feature
`client`, which adds the role `client` and the calls of a card that federates, sums `.text`,
`.rodata` and `.srodata` of the probe's object with `llvm-size`, and counts the symbols of
`core::sync::atomic` in each archive with `llvm-nm`. The builds are the proof that the engine uses
no atomic type, since the RV32IMC core has none. The objects of `compiler_builtins`, which the
archive carries for any Rust code on the target, are not the crate's and are printed apart.

It holds both images, the server image and the image with the roles `client` and `server`, so a card
may federate. The default features of `homa` keep the role `client`, so every server
and gateway on Linux keeps it, and the image of a card, built with `default-features = false`, takes
the roles its device card names.

In [ ]:
if half == "simulation":
    PROBE = P.tests / "fixtures" / "rs" / "homa-probe" / "Cargo.toml"
    PROBE_TARGET = "riscv32imc-unknown-none-elf"
    PROBE_DIR = P.target / "homa-probe"


    def build_probe(features, target_dir):
        extra = ["--features", features] if features else []
        run([CARGO, "build", "--locked", "--release", "--manifest-path", PROBE, "--target", PROBE_TARGET,
             "--target-dir", target_dir, *extra], cwd=P.root, timeout_s=build_timeout_s)
        return target_dir / PROBE_TARGET / "release" / "libhoma_probe.a"


    def object_sizes(archive):
        """Return the code and read only bytes of the probe's object and of the rest."""
        listing = run([LLVM_SIZE, "-A", archive], timeout_s=120).out
        sums = {"probe": 0, "compiler_builtins": 0}
        member = None
        for line in listing.splitlines():
            head = re.match(r"^(\S+\.o)\s+\(ex ", line)
            if head:
                member = "probe" if head.group(1).startswith("homa_probe") else "compiler_builtins"
                continue
            section = line.split()
            if member and section and re.match(r"^\.(text|rodata|srodata)", section[0]):
                sums[member] += int(section[1])
        return sums


    def atomic_symbols(archive):
        listing = run([LLVM_NM, "--demangle", archive], timeout_s=120, check=False).out
        return [line for line in listing.splitlines() if "core::sync::atomic" in line]


    whole_archive = build_probe(None, PROBE_DIR)
    frame_archive = build_probe("frame-only", PROBE_DIR / "frame-only")
    client_archive = build_probe("client", PROBE_DIR / "client")
    whole_sizes = object_sizes(whole_archive)
    frame_sizes = object_sizes(frame_archive)
    client_sizes = object_sizes(client_archive)
    atomics = atomic_symbols(whole_archive) + atomic_symbols(frame_archive) + atomic_symbols(client_archive)
    print("probe with the endpoint and AoFrame:", whole_sizes["probe"], "bytes of .text and .rodata")
    print("probe with AoFrame alone:           ", frame_sizes["probe"], "bytes")
    print("the engine beside AoFrame:          ", whole_sizes["probe"] - frame_sizes["probe"], "bytes")
    print("compiler_builtins of the target, not the crate's:", whole_sizes["compiler_builtins"], "bytes")
    largest = run([LLVM_NM, "--demangle", "--size-sort", "-S", whole_archive], timeout_s=120, check=False).out.splitlines()
    probe_symbols = [line for line in largest if " t " in line or " T " in line]
    print("the largest functions of the probe's object:")
    for line in [l for l in probe_symbols if "homa" in l][-12:]:
        print("  ", line)
    print("probe with the roles client and server:", client_sizes["probe"], "bytes,",
          client_sizes["probe"] - whole_sizes["probe"], "more than the server image")
    OWNER_Q12 = "the budget of 32 KiB that keeps the code of the client role, which federation needs"
    row("probe .text and .rodata, the server image, bytes", whole_sizes["probe"], "at most %d" % probe_budget,
        whole_sizes["probe"] <= probe_budget, beside=OWNER_Q12)
    row("probe .text and .rodata, client and server roles, bytes", client_sizes["probe"], "at most %d" % probe_budget,
        client_sizes["probe"] <= probe_budget,
        beside="a card that federates, %d bytes more than the server image" % (
            client_sizes["probe"] - whole_sizes["probe"]))
    row("symbols of core::sync::atomic in the probe", len(atomics), "0", not atomics)

### The table of section 6.6

Each profile fills the five rows of the table, the Homa column first and AoFrame alone second.
The bytes added are the probe's, the same on every profile since one image serves them all, and
the resident memory is the storage of the card of section 8.3 against the frame buffers of
AoFrame alone, one decoder and one encoder on each link.

In [ ]:
if half == "simulation":
    device_memory = sizes["storage_bytes"]["u8"] + aoframe_device
    complete = 0
    for name in PROFILES:
        cells = TABLE_66[name]
        cells["bytes added to the target"] = (whole_sizes["probe"], frame_sizes["probe"])
        cells["resident memory, bytes"] = (device_memory, aoframe_device)
        print(name)
        for label, (homa, alone) in cells.items():
            print("  %-28s %-16s %-16s" % (label, round(homa, 1) if isinstance(homa, float) else homa,
                                           round(alone, 1) if isinstance(alone, float) else alone))
        complete += all(value is not None for pair in cells.values() for value in pair) and len(cells) == 6
        span = SPAN[name]
        print("  every byte on the link, Homa: %d up for the pause, %d down for 16 KiB" % (
            span["link_bytes_pause_up"]["frame_bytes"], span["link_bytes_big"]["frame_bytes"]))
    print()
    row("rows of the table of section 6.6 complete, profiles", "%d of %d" % (complete, len(PROFILES)), "%d of %d" % (len(PROFILES), len(PROFILES)), complete == len(PROFILES))
    for name in ("Bluetooth GATT", "Serial Line"):
        homa_wait, base_wait = TABLE_66[name]["wait behind 16 KiB, ms"]
        speedup = base_wait / homa_wait if homa_wait else float("inf")
        row("small response behind 16 KiB on %s, times faster" % name, "%.1f" % speedup, "at least %d" % speedup_floor, speedup >= speedup_floor)
    row("lost requests under 1 percent loss on Serial Line, Homa", homa_lost, "0", homa_lost == 0)
    print("graphtp and graphtp-homa: the half graphtp of this notebook, an entry of the matrix of its own")

## The lints, the dependencies and the format

The crate `homa` sets its own edition 2021 and rust-version 1.85 and takes the lints of the
workspace unchanged, with `[lints]` and `workspace = true` and no table of its own. Since stage H2
its feature `unix` takes `socket2`, an optional dependency no other feature reaches, and `ip`,
`kernel-socket` and `helper` are declared ahead of their code and take it in their stages, so
`cargo metadata` lists `socket2` as the one dependency of `homa`, optional and reached by those four
features alone, and a build without them links no crate but `homa`. The format check runs `rustfmt`
of the pinned toolchain over the crate root with the crate's edition, which is the check `cargo fmt`
makes for that package, because `cargo fmt` takes no `--locked` and every cargo call of a notebook
passes it.

In [ ]:
if half == "simulation":
    with (P.root / "crates" / "homa" / "Cargo.toml").open("rb") as fh:
        manifest = tomllib.load(fh)
    with (P.root / "Cargo.toml").open("rb") as fh:
        workspace = tomllib.load(fh)
    print("homa [lints]:", manifest.get("lints"), " edition:", manifest["package"]["edition"],
          " rust-version:", manifest["package"]["rust-version"])
    print("workspace lints:", sorted(workspace["workspace"]["lints"]))
    lints_ok = manifest.get("lints") == {"workspace": True}
    row("homa inherits the workspace lints unchanged", manifest.get("lints"), "{'workspace': True}", lints_ok)

    metadata = json.loads(run([CARGO, "metadata", "--locked", "--format-version", "1"], cwd=P.root, timeout_s=timeout_s).out)
    homa = next(p for p in metadata["packages"] if p["name"] == "homa")
    gated = {feature: [item for item in homa["features"].get(feature, []) if item.startswith("dep:")]
             for feature in ("unix", "ip", "kernel-socket", "helper")}
    print("dependencies of homa:", [d["name"] for d in homa["dependencies"]])
    print("crates the four gated features take:", gated)
    GATED = {"unix", "ip", "kernel-socket", "helper"}
    ungated = [d["name"] for d in homa["dependencies"] if not d.get("optional")]
    reached = {d["name"]: sorted(f for f, items in homa["features"].items() if "dep:" + d["name"] in items)
               for d in homa["dependencies"]}
    print("features that reach each dependency:", reached)
    deps_ok = not ungated and all(set(features) <= GATED for features in reached.values())
    row("homa dependencies, unix, ip, kernel-socket, helper off", len(ungated), "0", deps_ok)

    formatted = run([RUSTFMT, "--check", "--edition", manifest["package"]["edition"], P.root / "crates" / "homa" / "src" / "lib.rs"],
                    cwd=P.root, timeout_s=timeout_s, check=False)
    print("rustfmt:", run([RUSTFMT, "--version"], cwd=P.root, timeout_s=60).out.strip())
    print("rustfmt --check exit:", formatted.returncode, formatted.out.strip()[:500])
    row("the format check on edition 2021", "exit %d" % formatted.returncode, "exit 0", formatted.returncode == 0)

## The client cells: the page's client role

The probe `tests/fixtures/rs/wasm-client-probe` is a module for `wasm32-unknown-unknown` with `homa`
in the features `graphtp-wasm` links, the role `client`, `alloc`, the compact form and both forms of
AoFrame. The probe links the standard library, whose allocator on `wasm32-unknown-unknown` serves
the feature `alloc`, since an allocator of the probe's own would need `unsafe`.

The cell builds the module with the release profile of the probe, size over speed with fat link time
optimization and no symbols, reads its size and its sections from the module bytes, counts the
entries of the import section, and prints the size under gzip at level 9 of Python's `gzip`. A
second build without the feature `stream-prefixed` prints the delta of that form, and a third build
that keeps the names of the functions prints what the code section holds, by the crate each function
comes from, so the owner reads where the bytes go.

> The row prints OPEN beside its measure and the gzip size, the verdict does not fail on it, every
> other row keeps its assert, and the row returns to an assert with the bound the owner names. The
> lead put three options to the owner: A (the lead's recommendation) judge the browser client on its
> compressed size at most 32 KiB, B raise the raw bound for the browser to 64 KiB, C keep 32 KiB raw
> and shrink the client role in a later batch.

So the row prints OPEN beside the bytes of the module and their size under gzip, whatever the
measure, and says beside it how far each stands from 32 KiB, which reads the options A, B and C
against the measure. The row of the imports keeps its assert, and the row of the bytes returns to
its assert with the bound the owner names.


In [ ]:
if half == "client":
    import gzip

    WASM_PROBE = P.tests / "fixtures" / "rs" / "wasm-client-probe" / "Cargo.toml"
    WASM_DIR = P.target / "wasm-client-probe"

    def build_wasm(target_dir, *extra):
        run([CARGO, "build", "--locked", "--release", "--manifest-path", WASM_PROBE, "--target",
             "wasm32-unknown-unknown", "--target-dir", target_dir, *extra], cwd=P.root, timeout_s=build_timeout_s)
        return (target_dir / "wasm32-unknown-unknown" / "release" / "wasm_client_probe.wasm").read_bytes()

    def leb(data, at):
        value, shift = 0, 0
        while True:
            byte = data[at]
            at += 1
            value |= (byte & 0x7F) << shift
            shift += 7
            if byte < 0x80:
                return value, at

    def sections(data):
        """Return the sections of a module as (id, body), after its magic and version."""
        assert data[:4] == b"\0asm" and data[4:8] == b"\x01\0\0\0", data[:8]
        out, at = [], 8
        while at < len(data):
            kind = data[at]
            size, at = leb(data, at + 1)
            out.append((kind, data[at:at + size]))
            at += size
        return out

    def imports(data):
        """Return the count of entries of the import section, 0 without one."""
        return sum(leb(body, 0)[0] for kind, body in sections(data) if kind == 2)

    def exports(data):
        names = []
        for kind, body in sections(data):
            if kind != 7:
                continue
            count, at = leb(body, 0)
            for _ in range(count):
                length, at = leb(body, at)
                names.append(body[at:at + length].decode())
                at += length + 1
                _, at = leb(body, at)
        return names

    def code_by_crate(data):
        """Return the bytes of the code section by the crate of each function, from the name section."""
        found = dict(sections(data))
        names = {}
        for kind, body in sections(data):
            if kind != 0:
                continue
            length, at = leb(body, 0)
            if body[at:at + length] != b"name":
                continue
            at += length
            while at < len(body):
                sub = body[at]
                size, at = leb(body, at + 1)
                end = at + size
                if sub == 1:
                    count, k = leb(body, at)
                    for _ in range(count):
                        index, k = leb(body, k)
                        length, k = leb(body, k)
                        names[index] = body[k:k + length].decode(errors="replace")
                        k += length
                at = end
        imported = imports(data)
        code = found[10]
        count, at = leb(code, 0)
        groups = {}
        for index in range(count):
            size, start = leb(code, at)
            name = names.get(index + imported, "")
            if "wasm_client_probe" in name or name.startswith("homa_probe"):
                crate = "the probe's own calls"
            elif "homa" in name:
                crate = "homa"
            elif "dlmalloc" in name or "__rust_alloc" in name or "__rust_realloc" in name or "__rust_dealloc" in name or "raw_vec" in name:
                crate = "the allocator of the standard library"
            else:
                crate = "the rest of core, alloc and std"
            groups[crate] = groups.get(crate, 0) + size + (start - at)
            at = start + size
        return groups

    module = build_wasm(WASM_DIR)
    plain = build_wasm(WASM_DIR / "no-prefixed", "--no-default-features")
    named = build_wasm(WASM_DIR / "names", "--config", 'profile.release.strip="none"')
    zipped = len(gzip.compress(module, compresslevel=9))
    print("module:", len(module), "bytes,", zipped, "under gzip, imports", imports(module), ", exports", exports(module))
    print("sections:", [(kind, len(body)) for kind, body in sections(module)])
    print("without stream-prefixed:", len(plain), "bytes, the delta of stream-prefixed", len(module) - len(plain), "bytes")
    groups = code_by_crate(named)
    print("the code section by crate, from a build that keeps the names:", groups)

    def against(size):
        """Say how far a size stands from the bound of P13."""
        if size == wasm_budget:
            return "at the bound"
        return "%d %s the bound" % (abs(size - wasm_budget), "above" if size > wasm_budget else "below")

    row("wasm client role, bytes raw and under gzip", "%d, gzip %d" % (len(module), zipped),
        "at most %d" % wasm_budget, len(module) <= wasm_budget, open_question=wasm_question,
        beside="%d bytes raw, %s, and %d under gzip, %s, the engine of homa %d of the code, the allocator %d%s" % (
            len(module), against(len(module)), zipped, against(zipped), groups.get("homa", 0),
            groups.get("the allocator of the standard library", 0),
            "" if wasm_question is None else
            ", OPEN under the lead's decision of 2026-09-28 until the owner answers owner question %d of the close page" % wasm_question))
    row("imports of the wasm client role, from the module bytes", imports(module), "0", imports(module) == 0)
    info("gzip size of the wasm client role, bytes", zipped, "no bound, decision P13 asks it printed")
    info("delta of stream-prefixed in the wasm client role, bytes", len(module) - len(plain), "no bound")


## The client cells: an answer in progress through `reply_reserve`

A listener that codes an answer through `reply` holds the whole answer in a buffer of the card's
longest answer and hands it to `reply`, which copies it into the pages of the RPC, so the answer is
in memory twice while it is in progress, and one that codes it through `reply_reserve` writes it
straight into the pages of the reservation and holds no buffer. The card's longest answer is 8,192
bytes.

The pages an answer of 8,192 bytes takes are measured on the engine: `homa-sim` runs one RPC whose
server holds a pool of pages of 512 bytes, once through `reply_reserve` and once through `reply`,
with a pool of exactly one request page and the pages of the answer, and again with one page fewer.
The server holds two slots, since a pool may give a slot 16 pages at most, and its LIMITS admit one
RPC at a time. A path that answers with the pool of the request and the answer and fails `NoSlot`
with one page fewer takes the pages of the answer and no page more, so the engine keeps no second
copy of the answer in either path. The buffer `reply` needs beside them is the whole answer, which
its signature takes. The probe for the device image calls both, `reply` and `reply_reserved`, whose
code the cell prints from the probe's object.


In [ ]:
if half == "client":
    page = 512
    answer_pages = reserve_answer // page
    RESERVE = {}
    for path_name, reserve in (("reply_reserve", True), ("reply", False)):
        for pages in (answer_pages + 1, answer_pages):
            got = sim([{
                "id": 0, "op": "run", "link": "loopback", "profile": {"name": "usb_serial"},
                "model": {"bytes_per_s": 1_000_000, "packets_per_event": 2, "latency_ns": 1_000_000},
                "config": {"client_slots": 8, "server_slots": 8, "page_len": 4096, "pages": 256, "max_gaps": 2},
                "config_b": {"client_slots": 0, "server_slots": 2, "page_len": page, "pages": pages},
                "limits_a": {"max_message": 65536}, "limits_b": {"max_rpcs": 1, "max_message": reserve_answer},
                "window": 1, "max_time_ns": 60 * 1_000_000_000,
                "rpcs": [{"len": 100, "response_len": reserve_answer, "reserve": reserve, "reserve_max": reserve_answer,
                          "end_at_ns": 50_000_000_000}],
            }], "reserve-%s-%d" % (path_name, pages))[0]
            rpc = got["rpcs"][0]
            RESERVE[(path_name, pages)] = {"outcome": rpc["outcome"], "reply_error": rpc["reply_error"]}
            print("%-13s a pool of %2d pages of %d bytes: %s" % (path_name, pages, page, RESERVE[(path_name, pages)]))
    reserved_pages_only = all(
        RESERVE[(name, answer_pages + 1)]["outcome"] == "ok" and RESERVE[(name, answer_pages)]["reply_error"] == "NoSlot"
        for name in ("reply_reserve", "reply"))
    in_progress = {"reply_reserve": answer_pages * page, "reply": answer_pages * page + reserve_answer}
    info("an answer in progress through reply_reserve, bytes", in_progress["reply_reserve"],
         "the pages of the answer alone, no buffer")
    info("an answer in progress through reply with a buffer, bytes", in_progress["reply"],
         "the pages and the answer buffer of %d bytes" % reserve_answer)
    row("reservation holds the answer's pages, no second copy", reserved_pages_only, "True",
        reserved_pages_only, beside="%d pages of %d bytes for an answer of %d bytes, the pool bound one page tight" % (
            answer_pages, page, reserve_answer))

    PROBE = P.tests / "fixtures" / "rs" / "homa-probe" / "Cargo.toml"
    PROBE_DIR = P.target / "homa-probe"
    run([CARGO, "build", "--locked", "--release", "--manifest-path", PROBE, "--target", PROBE_TARGET,
         "--target-dir", PROBE_DIR], cwd=P.root, timeout_s=build_timeout_s)
    archive = PROBE_DIR / PROBE_TARGET / "release" / "libhoma_probe.a"
    listing = run([LLVM_NM, "--demangle", "--size-sort", "-S", archive], timeout_s=120, check=False).out.splitlines()
    calls = {}
    for line in listing:
        cells = line.split()
        if len(cells) >= 4 and cells[2] in ("t", "T") and cells[3].startswith("homa_probe::reply"):
            calls[cells[3].split("::")[1]] = int(cells[1], 16)
    print("the code of the probe's calls, bytes:", calls)
    info("code of reply_reserved in the probe, bytes", calls.get("reply_reserved"), "no bound, beside reply at %s" % calls.get("reply"))


## The GraphTP rows on the build host

The half `graphtp` holds the budgets of `docs/graphtp-in-aosabi.md` section 8 that need no device.
The operation `card` of `aosabi-devtool graphtp` answers the bounds the listener of `aosabi-core`
holds for each card and makes the scratch of the card's encoder, whose bytes, the hash heads, the
chains, the tokens and the buffer of plain bytes of the window and its lookahead, are the memory an
answer in progress codes through. Beside it the cell prints the request buffer and the answer buffer
of each card and their sum with the scratch. Step 5 of `docs/graphtp-in-aosabi.md` section 3, no
plain copy of the whole answer on the device, is not met while `Server::answer` takes the whole
plain body.

The code the crates of the GraphTP stack hold for the ESP32-C3 is read by the object file: the build
of `aosabi-core` with its feature `homa` alone for `riscv32imc-unknown-none-elf` in the size profile
`player` of the workspace with fat LTO switched off, so each library holds machine code and not
bitcode, as `24_graphtp` builds the crate `graphtp`, in a target directory of its own, and the code
and read only data of the objects of `graphtp`, `graphtp-homa`, `homa`, `graphql-over-graphtp`, the
listener, and `aosabi-core` in it.

In [ ]:
if half == "graphtp":
    CARDS = ["esp32c3-devkitm1", "player", "test-graphtp", "server"]
    BUDGETS = {name: int(bound) for name, bound in (entry.split(":") for entry in encoder_budgets.split(","))}
    path = OUT / "cards.ndjson"
    path.write_text("".join(json.dumps({"id": n, "op": "card", "card": name}) + "\n" for n, name in enumerate(CARDS)),
                    encoding="utf-8")
    answers = [json.loads(line) for line in run([DEVTOOL, "graphtp", "--requests", path], cwd=P.root,
                                               timeout_s=timeout_s).out.splitlines()]
    CARD_LIMITS = {answer["card"]: answer for answer in answers}
    for name in CARDS:
        limits = CARD_LIMITS[name]
        coding = limits["coding"]
        print("%-17s heads %5d, tokens %5d, buffer %6d, window of 2^%d" % (
            name, coding["heads"], coding["tokens"], coding["buffer"], coding["window_bits"]))
    print()
    for name, bound in BUDGETS.items():
        limits = CARD_LIMITS[name]
        made = limits["scratch_made"]
        row("the encoder of %s, bytes" % name, made, "at most %d" % bound, made <= bound and made == limits["coding"]["scratch"])
    for name in CARDS:
        limits = CARD_LIMITS[name]
        info("the request and answer buffers of %s, bytes" % name,
             "%d and %d" % (limits["request_buffer"], limits["answer_buffer"]), "the card's bounds", "DOCUMENTED")
        info("the scratch and both buffers of %s, bytes" % name,
             limits["scratch_made"] + limits["request_buffer"] + limits["answer_buffer"],
             "a sum of the card's bounds without the listener's copies of the content and the plain body",
             "DOCUMENTED")

    GRAPHTP_DIR = P.target / "graphtp-budget"
    built = run([CARGO, "build", "--locked", "-p", "aosabi-core", "--no-default-features", "--features", "homa",
                 "--profile", "player", "--config", "profile.player.lto=false", "--target", PROBE_TARGET,
                 "--target-dir", GRAPHTP_DIR], cwd=P.root, timeout_s=build_timeout_s, check=False)
    print("cargo build --locked -p aosabi-core --features homa --profile player for %s: exit %d, %.1f s" % (
        PROBE_TARGET, built.returncode, built.duration_s))
    row("the GraphTP stack builds for %s" % PROBE_TARGET, "exit %d" % built.returncode, "exit 0", built.returncode == 0)
    DEPS = GRAPHTP_DIR / PROBE_TARGET / "player" / "deps"
    for crate in ("graphtp", "graphtp_homa", "homa", "graphql_over_graphtp", "aosabi_core"):
        archives = sorted(DEPS.glob("lib%s-*.rlib" % crate))
        total = 0
        for archive in archives:
            listing = run([LLVM_SIZE, "-A", archive], timeout_s=120).out
            for line in listing.splitlines():
                parts = line.split()
                if parts and re.match(r"^\.(text|rodata|srodata)", parts[0]):
                    total += int(parts[1])
        info("%s for %s by the object file, bytes" % (crate.replace("_", "-"), PROBE_TARGET), total,
             "%d rlib, before a card's link and without the generic code a card instantiates" % len(archives))

## The LoRa probe cell

The HT-CT62 is an ESP32-C3 with an SX1262. The cell builds the device probe
`tests/fixtures/rs/homa-probe` for its RV32IMC core with the features of the HT-CT62 one part at a
time, each build the one before it and one part more, and reads the part as the difference of the
two builds' `.text`, `.rodata` and `.srodata` of the probe's object: `pacer-budget`, the image of
the ESP32-C3 with the budget pacer the airtime ledger runs over, which is `homa` with `pacer-budget`
and AoFrame, then `aead`, which adds `chacha20poly1305` through the seal and the open of the
carrier, `lora-frame`, which adds the frame, the window, the counters, the ledger and the time on
air called on their own, `lora`, which adds the carrier over a radio of the probe and the turn of a
card's driver over it through the same transmit every link of the card takes its packets through,
and `sx1262`, which puts the driver of the SX1262 over a HAL of the probe in the place of that
radio, with a TCXO over DIO3 and an RF switch over DIO2, so every command of its configuration
counts. LoRa is everything the last build holds beyond the first, the carrier's own code among it.
Every build must hold no symbol of `core::sync::atomic`.

In [ ]:
if half == "lora":
    PROBE = P.tests / "fixtures" / "rs" / "homa-probe" / "Cargo.toml"
    PROBE_TARGET = "riscv32imc-unknown-none-elf"
    LORA_DIR = P.target / "homa-probe-lora-cell"


    def build_lora_probe(features):
        target_dir = LORA_DIR / features
        run([CARGO, "build", "--locked", "--release", "--manifest-path", PROBE, "--target", PROBE_TARGET,
             "--target-dir", target_dir, "--features", features], cwd=P.root, timeout_s=build_timeout_s)
        return target_dir / PROBE_TARGET / "release" / "libhoma_probe.a"


    def probe_part(archive):
        """Return the code and read only bytes of the probe's object in `archive`."""
        listing = run([LLVM_SIZE, "-A", archive], timeout_s=120).out
        total, member = 0, None
        for line in listing.splitlines():
            head = re.match(r"^(\S+\.o)\s+\(ex ", line)
            if head:
                member = head.group(1).startswith("homa_probe")
                continue
            section = line.split()
            if member and section and re.match(r"^\.(text|rodata|srodata)", section[0]):
                total += int(section[1])
        return total


    LORA_BUILDS = {}
    atomics = []
    for features in ("pacer-budget", "aead", "lora-frame", "lora", "sx1262"):
        archive = build_lora_probe(features)
        LORA_BUILDS[features] = probe_part(archive)
        listing = run([LLVM_NM, "--demangle", archive], timeout_s=120, check=False).out
        atomics += [line for line in listing.splitlines() if "core::sync::atomic" in line]
        print("probe with %-14s %6d bytes of .text and .rodata" % (features, LORA_BUILDS[features]))
    homa_part = LORA_BUILDS["pacer-budget"]
    aead_part = LORA_BUILDS["aead"] - LORA_BUILDS["pacer-budget"]
    frame_part = LORA_BUILDS["lora-frame"] - LORA_BUILDS["aead"]
    carrier_part = LORA_BUILDS["lora"] - LORA_BUILDS["lora-frame"]
    sx1262_part = LORA_BUILDS["sx1262"] - LORA_BUILDS["lora"]
    lora_part = LORA_BUILDS["sx1262"] - LORA_BUILDS["pacer-budget"]
    print()
    OWNER_Q12 = "the budget of 32 KiB that keeps the code of the client role, which federation needs"
    row("homa with pacer-budget and AoFrame, the HT-CT62, bytes", homa_part, "at most %d" % lora_homa_budget,
        homa_part <= lora_homa_budget,
        beside="the lead's decision of 2026-09-28 gives the row the %d bytes of the owner's answer to owner"
               " question 12, %s, since the HT-CT62 carries an ESP32-C3, and answers owner question %d with its"
               " option A, which the owner may change, the measure is %d bytes %s that bound and %d bytes %s the"
               " plan's 14.5 KiB, %d bytes, the ESP32-C3's 13.5 KiB and 1 KiB of pacer-budget" % (
                   lora_homa_budget, OWNER_Q12, lora_question,
                   abs(homa_part - lora_homa_budget), "above" if homa_part > lora_homa_budget else "below",
                   abs(homa_part - lora_plan_bound), "above" if homa_part > lora_plan_bound else "below",
                   lora_plan_bound))
    row("LoRa added to the HT-CT62 image, bytes", lora_part, "at most %d" % lora_budget, lora_part <= lora_budget)
    row("the driver of the SX1262, bytes", sx1262_part, "at most %d" % sx1262_budget, sx1262_part <= sx1262_budget)
    row("the frame, the ledger and the time on air, bytes", frame_part, "at most %d" % lora_frame_budget,
        frame_part <= lora_frame_budget)
    row("chacha20poly1305 with its companions, bytes", aead_part, "at most %d" % aead_budget, aead_part <= aead_budget)
    row("symbols of core::sync::atomic in the LoRa probes", len(atomics), "0", not atomics)
    info("the carrier and the turn of a card's driver, bytes", carrier_part, "in the LoRa row, no bound of its own")
    sizes_path = OUT / "lora-sizes.ndjson"
    sizes_path.write_text('{"id":0,"op":"sizes"}\n', encoding="utf-8")
    lora_sizes = json.loads(run([DEVTOOL, "lora", "--requests", sizes_path], cwd=P.root, timeout_s=timeout_s).out)
    info("the state of the carrier with its ledger, bytes on this host", lora_sizes["carrier"] + lora_sizes["ledger_buckets"],
         "the design note's section 8.3 gives 808 on the card, ASSUMED, and the host's pointers are twice the card's")
    info("graphtp", "not built", "the GraphTP track's crate, which batch G3 links into a device image", mark="DOCUMENTED")
    info("graphtp-homa", "not built", "the GraphTP track's crate, which batch G3 links into a device image", mark="DOCUMENTED")

## The raw server cell

The raw server cell of stage H4 weighs a server build that names GraphTP(Homa) on Network over raw
IP 146, the example `raw_server` of `aosabi-carrier-network`, which serves the listener over the IP
driver of `homa::ip` in front of a fetch that answers at once and ships in no artifact, and reads
the state of the engine of the Network profile and of the helper `homa-raw` through
`aosabi-devtool homa-ip` and its operation `sizes`.

The plan names the release build of the devtool's `homa-ip` server as the input. The devtool links
`homa` with the features of every primitive it serves, the simulators, Unix Socket, the compact
form and both roles among them, and the driver beside loops of its own, so the sums of its symbols
would charge a server build with the code of the devtool. The example links what a server card that
names GraphTP(Homa) on Network turns on and nothing else, so it stands in for that build.

The rows of the bytes read the plan's measure. The example is built with the release profile of a
server card, fat link time optimization and one code unit, and `llvm-nm` gives the code and read only
bytes of the symbols each crate names. Fat link time optimization folds the engine's constructors
and the opening of the raw socket into the driver's one asynchronous block, so the sum of the
driver holds them. The same build with the line tables kept gives every byte of code the source file
it comes from, which charges each crate its own lines alone, code without read only data, and the
cell prints those bytes beside the sums of that build's symbols, which show whether the line tables
move the code.

The rows of the memory read the bytes of each part of the storage of a server card's endpoint, 4,096
peers and 16,384 RPCs with a pool of 268 pages of 62,500 bytes, as the operation `sizes` answers
them. A peer is the place of an address with its Homa port, its cutoffs and its share of the buckets
of the peer table. An RPC without its pages is its slot, the words of its 16 gaps, its share of the
buckets of the slot table and its items in the transmit heap and the grant heap. The state of the
helper's client role is read the same way with the configuration its source writes, and printed.

So the entry `23_homa_budget-raw-server` names both questions, and each row prints OPEN beside its
measure, whatever the measure. With a question None, a row that crosses its bound prints CROSSED and
fails the entry. The bytes of `socket2`, the Network carrier's own, and the size of the helper
`homa-raw` are printed with no bound.

In [ ]:
if half == "raw-server":
    import collections

    RAW_DIR = P.target / "raw-server-cell"
    binary = RAW_DIR / "release" / "examples" / "raw_server"

    def build_example(label, extra):
        """Build the example raw_server with the release profile and `extra`."""
        build = run([CARGO, "build", "--locked", "--release", *extra, "-p", "aosabi-carrier-network", "--features",
                     "homa-ip", "--example", "raw_server", "--target-dir", RAW_DIR],
                    cwd=P.root, timeout_s=build_timeout_s, check=False)
        print("cargo build --locked --release%s --example raw_server: exit %d, %.1f s" % (
            label, build.returncode, build.duration_s))
        assert build.returncode == 0, build.err[-2000:]

    LLVM_DWARFDUMP = shutil.which("llvm-dwarfdump")
    assert LLVM_DWARFDUMP, "the raw server cell reads the line tables with llvm-dwarfdump"

    def crate_of(path):
        """Return the crate a source path belongs to."""
        for pattern in (r"/crates/([^/]+)/", r"/library/([^/]+)/", r"/(?:registry/src/[^/]+|rust/deps)/([^/]+?)-[0-9][^/]*/",
                        r"/(?:vendor|third_party/rust)/([^/]+)/"):
            found = re.search(pattern, path)
            if found:
                return found.group(1).replace("-", "_")
        return "(other)"

    def line_bytes(path):
        """Return the bytes of code of each crate by the source file of each byte."""
        units = run([LLVM_DWARFDUMP, "--debug-info", "--recurse-depth=0", path], timeout_s=300).out
        comp_dirs, offset = {}, None
        for line in units.splitlines():
            found = re.search(r"DW_AT_stmt_list\s+\((0x[0-9a-f]+)\)", line)
            if found:
                offset = int(found.group(1), 16)
            found = re.search(r'DW_AT_comp_dir\s+\("(.*)"\)', line)
            if found and offset is not None:
                comp_dirs[offset] = found.group(1)
                offset = None
        listing = run([LLVM_DWARFDUMP, "--debug-line", path], timeout_s=300).out
        sums = collections.Counter()
        dirs, files, current, previous = {}, {}, None, None
        for line in listing.splitlines():
            found = re.match(r"debug_line\[(0x[0-9a-f]+)\]", line)
            if found:
                dirs = {0: comp_dirs.get(int(found.group(1), 16), "")}
                files, current, previous = {}, None, None
                continue
            found = re.match(r'include_directories\[\s*(\d+)\] = "(.*)"', line)
            if found:
                dirs[int(found.group(1))] = found.group(2)
                continue
            found = re.match(r"file_names\[\s*(\d+)\]:", line)
            if found:
                current = int(found.group(1))
                files[current] = ["", 0]
                continue
            found = re.match(r'\s+name: "(.*)"', line)
            if found and current is not None:
                files[current][0] = found.group(1)
                continue
            found = re.match(r"\s+dir_index: (\d+)", line)
            if found and current is not None:
                files[current][1] = int(found.group(1))
                continue
            found = re.match(r"0x([0-9a-f]{16})\s+\d+\s+\d+\s+(\d+)\s+\d+\s+\d+\s*\d*\s*(.*)", line)
            if found:
                address, file_index, flags = int(found.group(1), 16), int(found.group(2)), found.group(3)
                if previous is not None:
                    start, index = previous
                    name, directory = files.get(index, ["", 0])
                    full = name if name.startswith("/") else dirs.get(directory, "") + "/" + name
                    if not full.startswith("/"):
                        full = dirs.get(0, "") + "/" + full
                    sums[crate_of(full)] += address - start
                previous = None if "end_sequence" in flags else (address, file_index)
        return sums

    def symbol_bytes(path):
        """Return the code and read only bytes of each crate by the names of its symbols."""
        listing = run([LLVM_NM, "--print-size", "--size-sort", "--demangle", path], timeout_s=120).out
        sums = collections.Counter()
        for line in listing.splitlines():
            parts = line.split(None, 3)
            if len(parts) < 4 or parts[2].lower() not in ("t", "r"):
                continue
            head = re.match(r"([A-Za-z_][A-Za-z0-9_]*)::", parts[3].lstrip("<&"))
            sums[head.group(1) if head else "(other)"] += int(parts[1], 16)
        return sums

    # The plan's measure: the release build of a server card and the code and
    # read only bytes of the symbols each crate names.
    build_example("", [])
    by_symbol = symbol_bytes(binary)
    plain_size = binary.stat().st_size
    # The side measure: the same build with the line tables kept, read by the
    # source file of each byte of code, beside the sums of its own symbols.
    build_example(", line tables kept,", ["--config", 'profile.release.debug="line-tables-only"'])
    by_line = line_bytes(binary)
    by_symbol_lined = symbol_bytes(binary)
    print("the raw server, %d bytes of release binary, and its crates by symbol, by symbol with the line tables kept"
          " and by source line:" % plain_size)
    for name in sorted(set(by_symbol) | set(by_line), key=lambda name: -by_symbol.get(name, 0))[:16]:
        print("  %-28s %8d %8d %8d" % (name, by_symbol.get(name, 0), by_symbol_lined.get(name, 0), by_line.get(name, 0)))
    homa_symbol, network_symbol = by_symbol["homa"], by_symbol["aosabi_carrier_network"]
    row("homa with the IP driver, bytes by symbol", homa_symbol, "at most %d" % homa_ip_budget,
        0 < homa_symbol <= homa_ip_budget)
    row("the Network driver aosabi-carrier-network, bytes by symbol", network_symbol,
        "at most %d" % network_driver_budget, 0 < network_symbol <= network_driver_budget,
        open_question=driver_question,
        beside="%d bytes %s the bound, the engine's constructors and the opening of the raw socket folded into"
               " the driver's asynchronous block among them, owner question 29" % (
                   abs(network_symbol - network_driver_budget),
                   "above" if network_symbol > network_driver_budget else "below"))
    lined_same = all(by_symbol_lined.get(name, 0) == by_symbol.get(name, 0)
                     for name in ("homa", "aosabi_carrier_network", "socket2"))
    info("the line tables kept, homa and the driver by symbol",
         "%d and %d" % (by_symbol_lined["homa"], by_symbol_lined["aosabi_carrier_network"]),
         "equal to the release build" if lined_same else "the line tables move the code")
    info("homa with the IP driver, bytes of code by source line", by_line["homa"],
         "its own lines alone, code without read only data")
    info("aosabi-carrier-network, bytes of code by source line", by_line["aosabi_carrier_network"],
         "its own lines alone, code without read only data")
    info("socket2, the carrier's own, bytes by symbol and by line",
         "%d and %d" % (by_symbol["socket2"], by_line["socket2"]), "the carrier's own, no bound")
    helper = run([CARGO, "build", "--locked", "--release", "-p", "homa", "--features", "helper", "--bin", "homa-raw",
                  "--target-dir", RAW_DIR], cwd=P.root, timeout_s=build_timeout_s, check=False)
    assert helper.returncode == 0, helper.err[-2000:]
    helper_bin = RAW_DIR / "release" / "homa-raw"
    sections = run([LLVM_SIZE, "-A", helper_bin], timeout_s=120).out
    helper_code = sum(int(parts[1]) for parts in (line.split() for line in sections.splitlines())
                      if parts and parts[0] in (".text", ".rodata"))
    info("the helper homa-raw, bytes of its binary and of its code",
         "%d and %d" % (helper_bin.stat().st_size, helper_code),
         "the release binary and its .text and .rodata, printed, no bound before the first run")

    # The memory of the engine of Network, by the parts of its storage.
    def storage(answer):
        """Return the bytes of the peers, of the RPCs without their pages and
        of the pool of `answer`, each part of the storage counted once."""
        types, parts = answer["types"], answer["parts"]
        peers = {"place": parts["peers"] * types["peer"], "cutoffs": parts["peer_prio"] * types["peer_prio"],
                 "table": parts["peer_index"] * types["bucket"]}
        rpcs = {"slot": parts["slots"] * types["slot"], "gaps": parts["gaps"] * types["gap"],
                "table": parts["slot_index"] * types["bucket"],
                "heaps": (parts["tx_heap"] + parts["grant_heap"]) * types["item"]}
        rest = parts["pages"] + parts["sizes"] * types["size"]
        assert sum(peers.values()) + sum(rpcs.values()) + rest == answer["storage_bytes"], answer
        return peers, rpcs, parts["pages"]

    sizes_path = OUT / "raw-server-sizes.ndjson"
    HELPER_SOURCE = (P.root / "crates" / "homa" / "src" / "bin" / "homa_raw.rs").read_text(encoding="utf-8")
    block = re.search(r"let config = Config \{(.*?)\};", HELPER_SOURCE, re.S)
    assert block, "the helper's source writes its configuration in one Config block"
    helper_config = {key: int(value.replace("_", "")) for key, value in
                     re.findall(r"\b(peers|client_slots|server_slots|max_gaps): ([0-9_]+),", block.group(1))}
    page_len = int(re.search(r"const PAGE_LEN: u32 = ([0-9_]+);", HELPER_SOURCE).group(1).replace("_", ""))
    pool_shift = re.search(r"const POOL: u32 = ([0-9]+) << ([0-9]+);", HELPER_SOURCE)
    helper_ports = int(re.search(r"let mut ports = \(([0-9_]+)u16, ([0-9_]+)u16\);", HELPER_SOURCE)
                       .group(2).replace("_", ""))
    assert sorted(helper_config) == ["client_slots", "max_gaps", "peers", "server_slots"], helper_config
    assert "pages: PAGES," in block.group(1) and "const PAGES: u32 = POOL / (PAGE_LEN + 2);" in HELPER_SOURCE
    helper_config.update(page_len=page_len, pages=(int(pool_shift.group(1)) << int(pool_shift.group(2))) // (page_len + 2))
    sizes_path.write_text(json.dumps({"id": 0, "op": "sizes"}) + "\n"
                          + json.dumps({"id": 1, "op": "sizes", "config": helper_config}) + "\n", encoding="utf-8")
    answered = run([DEVTOOL, "homa-ip", "--requests", sizes_path], cwd=P.root, timeout_s=timeout_s, check=False)
    answers = [json.loads(line) for line in answered.out.splitlines()]
    if answered.returncode != 0 or len(answers) != 2:
        PANICS.append("homa-ip sizes: exit %d, %d answers to 2 requests, %s" % (
            answered.returncode, len(answers), answered.err.strip()[-300:]))
    server, helper_sizes = answers
    print("the server card's endpoint: %d peers, %d RPCs, %d pages of %d bytes, %d gaps a message, types %s" % (
        server["peers"], server["server_slots"], server["pages"], server["page_len"], server["max_gaps"],
        json.dumps(server["types"])))
    peers, rpcs, pool = storage(server)
    peer = sum(peers.values()) / server["peers"]
    rpc = sum(rpcs.values()) / server["server_slots"]
    row("a peer of Network with its cutoffs and table, bytes", "%g" % peer,
        "at most %d" % network_peer_budget, peer <= network_peer_budget,
        beside="%s bytes of its place, an address with its Homa port, %s of its cutoffs and %s of the buckets of the peer table" % tuple(
            "%g" % (peers[k] / server["peers"]) for k in ("place", "cutoffs", "table")))
    each = {k: v / server["server_slots"] for k, v in rpcs.items()}
    row("an RPC of Network without its pages, bytes", "%g" % rpc, "at most %d" % network_rpc_budget,
        rpc <= network_rpc_budget, open_question=rpc_question,
        beside="%g bytes of its slot, %g of the words of its %d gaps, %g of the buckets of the slot table and %g of"
               " its items in the two heaps, %g without the gaps, owner question 32" % (
                   each["slot"], each["gaps"], server["max_gaps"], each["table"], each["heaps"], rpc - each["gaps"]))
    state = sum(peers.values()) + sum(rpcs.values())
    info("%d peers and %d RPCs without their pages, bytes" % (server["peers"], server["server_slots"]), state,
         "%.2f MiB beside the plan's about 4.2 MiB" % (state / 2 ** 20))
    info("the server card's page pool with its page table, bytes", pool,
         "%d pages of %d bytes, inside the plan's 16 MiB, %d bytes, which the card may lower" % (
             server["pages"], server["page_len"], 16 << 20))
    info("the buffers of a served path beside its endpoint, bytes", server["path_buffers"],
         "the task's receive buffer and the socket's datagram buffer")
    h_peers, h_rpcs, h_pool = storage(helper_sizes)
    h_state = sum(h_peers.values()) + sum(h_rpcs.values())
    info("the helper's %d peers and %d RPCs without pages, bytes" % (
        helper_sizes["peers"], helper_sizes["client_slots"]), h_state,
         "its client role, one engine the connections of its port range share, whatever --ports names")
    info("the helper's page pool with its page table, bytes", h_pool,
         "shared, never grown by --ports, the engine and the pool %d bytes a connection of the default %d ports"
         % ((h_state + h_pool) // helper_ports, helper_ports))

## The cells of later stages

The halves this entry does not name run as the matrix entries of their own.

In [ ]:
OTHER = {"simulation": ("simulation half and the probe cell", "23_homa_budget-simulation"),
         "client": ("client cells", "23_homa_budget-client"),
         "graphtp": ("GraphTP rows on the build host", "23_homa_budget-graphtp"),
         "lora": ("LoRa probe cell", "23_homa_budget-lora"),
         "raw-server": ("raw server cell", "23_homa_budget-raw-server")}
for other, (name, entry) in OTHER.items():
    if other != half:
        print("not run: %s, which runs as its own entry of the matrix, %s" % (name, entry))
for cell, stage in (("kernel cells", "H5"),
                    ("zero byte assertion", "H7"), ("image half", "DC9 and round 3 of M12")):
    print("not run: %s, which waits for %s" % (cell, stage))
if half == "graphtp":
    print("not run: the check that the worker build links no graphtp, graphtp-homa or homa, a row of"
          " 08_card_builds of M5, which waits for the Worker build of M5")

## Verdict

Every row of the exit number prints its measured value beside its bound, and a row that crosses its
bound prints CROSSED. The notebook fails on any crossed row, the rows of a budget included, since a
crossed bound fails the change that crosses it and the owner decides whether the bound moves. The
owner answered both questions, so no row of the simulation half prints OPEN. A measure printed as
information has no bound and no assert.

> The row prints OPEN beside its measure and the gzip size, the verdict does not fail on it, every
> other row keeps its assert, and the row returns to an assert with the bound the owner names. The
> lead put three options to the owner: A (the lead's recommendation) judge the browser client on its
> compressed size at most 32 KiB, B raise the raw bound for the browser to 64 KiB, C keep 32 KiB raw
> and shrink the client role in a later batch.

So the client cells pass with that row OPEN and fail on any other row that crosses its bound, and
the row asserts again once the owner names its bound.

A measure above 32 KiB prints CROSSED and fails the entry `23_homa_budget-lora`, and the owner may
name another bound for the row in place of the lead's.

The rows of the raw server cell judge the sums of the symbols of the release build, as the plan
reads them, and the memory of a peer and of an RPC without its pages.

> The owner may still choose B, C or D.

So the raw server cell passes with both rows OPEN through the parameters `driver_question` 29 and
`rpc_question` 32 of its entry, and fails on any other row that crosses its bound. A row whose
question is None asserts again, and an answer B, C or D of the owner moves the bound, the measure or
the code its question names.

In [ ]:
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
print()
print("%-58s %-18s %-18s %s" % ("row", "measured", "bound", "state"))
for name, value, bound, state, mark, beside in ROWS:
    print("%-58s %-18s %-18s %s" % (name[:58], value, bound, state))
    if beside:
        print("  beside: %s" % beside)
print()
print("information, no bound:")
for name, value, note, mark in INFO:
    print("%-58s %-18s %s, %s" % (name[:58], value, note, mark))
crossed = [name for name, _, _, state, _, _ in ROWS if state == "CROSSED"]
opened = ["%s %s" % (name, state[len("OPEN "):]) for name, _, _, state, _, _ in ROWS if state.startswith("OPEN")]
held = [name for name, _, _, state, _, _ in ROWS if state == "ok"]
print()
what = {"simulation": "the simulation half and the probe cell", "client": "the client cells",
        "graphtp": "the GraphTP rows on the build host", "lora": "the LoRa probe cell",
        "raw-server": "the raw server cell"}[half]
print("VERDICT: %d of %d rows of %s hold, %d OPEN, %d CROSSED%s%s" % (
    len(held), len(ROWS), what, len(opened), len(crossed),
    "" if not opened else ". OPEN: " + "; ".join(opened),
    "" if not crossed else ". CROSSED: " + "; ".join(crossed)))
assert not PANICS, PANICS
assert not crossed, crossed